# Restaurant Closure Analysis — Extraction, Cleaning & Load

Source data: [Yelp Open Dataset](https://business.yelp.com/data/resources/open-dataset/)

This notebook extracts `business.json` from the ~5GB Yelp archive, converts it to CSV, cleans and reduces columns, extracts price range from the nested `attributes` field, and loads the result into PostgreSQL.

**Setup:** copy `.env.example` to `.env` and fill in your own local Postgres credentials before running the load cell. Never commit `.env` — it's already in `.gitignore`.

In [ ]:
# 1. Extract business.json from the Yelp tar archive
# (the archive is too large to fully unpack at once)
import tarfile
import os
import shutil

tar_path = r"data/raw/yelp_dataset.tar"           # path to the downloaded Yelp archive
output_dir = r"data/raw/business_extracted"
output_file = os.path.join(output_dir, "business.json")

os.makedirs(output_dir, exist_ok=True)

with tarfile.open(tar_path, "r:*") as tar:
    member = next((m for m in tar.getmembers() if m.name.endswith("business.json")), None)
    if member:
        with tar.extractfile(member) as src, open(output_file, "wb") as dst:
            shutil.copyfileobj(src, dst)
        print(f"Extracted successfully to:\n{output_file}")
    else:
        print("business.json not found in the archive.")

In [ ]:
# 2. Convert business.json (newline-delimited JSON) to CSV
import json
import csv

json_file = r"data/raw/business_extracted/business.json"
csv_file = r"data/raw/business_extracted/business.csv"

with open(json_file, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

keys = data[0].keys()

with open(csv_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=keys)
    writer.writeheader()
    writer.writerows(data)

print("CSV saved successfully!")

In [ ]:
# 3. Drop unneeded columns and extract price_range from the
#    nested `attributes` text field via regex, with a safe default
import csv
import re

input_path = r"data/raw/business_extracted/business.csv"
output_path = r"data/processed/business_clean.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

keep_columns = [
    'business_id', 'name', 'city', 'state',
    'stars', 'review_count', 'is_open', 'categories'
]

def extract_price_range(attributes_text):
    if not attributes_text or attributes_text.strip() == '':
        return 2
    patterns = [
        r'"RestaurantsPriceRange2"\s*:\s*"(\d)"',
        r"'RestaurantsPriceRange2'\s*:\s*'(\d)'",
        r'"RestaurantsPriceRange2"\s*:\s*(\d)',
    ]
    for pattern in patterns:
        match = re.search(pattern, attributes_text)
        if match:
            price = int(match.group(1))
            if 1 <= price <= 4:
                return price
    return 2

with open(input_path, 'r', encoding='utf-8') as infile, \
     open(output_path, 'w', newline='', encoding='utf-8') as outfile:

    reader = csv.DictReader(infile)
    writer = csv.DictWriter(outfile, fieldnames=keep_columns + ['price_range'])
    writer.writeheader()

    row_count = 0
    price_counts = {1: 0, 2: 0, 3: 0, 4: 0, 'missing': 0}

    for row in reader:
        price = extract_price_range(row.get('attributes', ''))
        price_counts[price if price in [1, 2, 3, 4] else 'missing'] += 1

        clean_row = {col: row.get(col, '') for col in keep_columns}
        clean_row['price_range'] = price
        writer.writerow(clean_row)
        row_count += 1

print(f"Cleaned {row_count:,} rows -> {output_path}")
print("\nPrice range distribution:")
for p in [1, 2, 3, 4, 'missing']:
    count = price_counts[p]
    pct = (count / row_count) * 100
    symbol = {1: '$', 2: '$$', 3: '$$$', 4: '$$$$', 'missing': '??'}.get(p, '?')
    print(f"  {p} ({symbol}): {count:,} ({pct:.1f}%)")

In [ ]:
# 4. Load into PostgreSQL
# Credentials are read from environment variables (via a local .env
# file + python-dotenv), never hardcoded here.
import os
import csv
import psycopg2
from dotenv import load_dotenv

load_dotenv()

conn = psycopg2.connect(
    dbname=os.environ["DB_NAME"],
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    host=os.environ.get("DB_HOST", "localhost"),
)
cursor = conn.cursor()

csv_path = r"data/processed/business_clean.csv"

with open(csv_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    row_count = 0
    for row in reader:
        cursor.execute(
            """
            INSERT INTO restaurants
            (business_id, name, city, state, stars, review_count, is_open, price_range, categories)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (business_id) DO NOTHING
            """,
            (
                row['business_id'], row['name'], row['city'], row['state'],
                row['stars'], row['review_count'], row['is_open'],
                row['price_range'], row['categories'],
            ),
        )
        row_count += 1

conn.commit()
print(f"Loaded {row_count} rows into restaurants")
cursor.close()
conn.close()